# Search for DOIs using title search in OpenAlex
This notebook searches for DOIs for IPCC references. Each title is passed to the OpenAlex API to search in the title. 
First returns are taken as they are assumed to have the highest score. 
Levenshtein distance to be performed on two titles to confirm match. <br>
 - used 3.12 global

In [1]:
# import libraries
import pandas as pd
import os
import requests
import pickle
import json
from colorama import Fore,Back,Style
import time
import csv
from tqdm import tqdm


# set display options
#pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_columns', None) # show all columns
pd.set_option('display.max_colwidth', None)  # Show full content of each column


In [10]:
# import txt file with one reference per line
file = "titles_with_no_DOIS.txt"
df = pd.read_csv(file,encoding='utf-8',sep="\t")

# separate from first ": " and next ". " keep what is between these two. Create new column
def extract_title(x:str)->str:
    start:int = x.find(': ')
    if start != -1:
        start += 2
    end:int = x.find('. ', start)
    return x[start:end]

# extract all the titles
df['title'] = df['Titles with no DOIS'].apply(extract_title)

# string lower and replace punct with space
import string

df['title'] = df['title'].apply(lambda x: x.lower().translate(str.maketrans('','',string.punctuation)))

In [52]:

"""
sends an API call to the OpenAlex REST API
returns full metadata
"""
def get_openalex_data(title:str):
    API_URL = f'https://api.openalex.org/works?filter=title.search:"{title}"'
    #make API call 
    try:
        time.sleep(0.2)
        response = requests.get(API_URL)
        response.raise_for_status()

        data = response.json()
        #print(data)
        count = data.get('meta').get('count')
        if count >0:
            results = data.get('results')[0]
            oa_id = data.get('results')[0].get('id')
            oa_title = data.get('results')[0].get('title')
            print(f"oa_id: {oa_id}\noa_title: {oa_title}\n")
            stuff = {'count':count,
                'oa_id':oa_id,
                'oa_title':oa_title}
        
        else:
            stuff = None

        return stuff

    except requests.exceptions.HTTPError as httperror:
        print(f"HTTP error: [httperror]")
        return 0
    except requests.exceptions.ConnectionError as conny:
        print(f"Connection error: {conny}")
        return 
    except requests.exceptions.Timeout as timey:
        print(f"timeout error: {timey}")
        return 0
    except requests.exceptions.RequestException as reqy:
        print(f"request error: {reqy}")
        return 0

"""

    data = {'doi':doi,
            'doi_type':doi_type,
            'title':title,
            'abstract':abstract,
            'citedby_count':citedby_count,
            'doi_url':doi_url,
            }

    # time delay for rate limiting
    time.sleep(1)

    return data

"""
title = "knowing our lands and resources indigenous and local knowledge of biodiversity and ecosystem services in the americas"
get_openalex_data(title)

df_results = df['title'].apply(get_openalex_data)
print(Fore.LIGHTMAGENTA_EX + 'done')


oa_id: https://openalex.org/W1523078206
oa_title: Soil and human security in the 21st century

oa_id: https://openalex.org/W2070054180
oa_title: Racial/ethnic residential segregation: Framing the context of health risk and health disparities

oa_id: https://openalex.org/W2561224299
oa_title: AMAP assessment 2015: human health in the Arctic

oa_id: https://openalex.org/W2788628873
oa_title: Arctic carbon Cycling. Snow, Water, Ice and Permafrost in the Arctic (SWIPA) 2017

oa_id: https://openalex.org/W2976369162
oa_title: The Future of the Law of the Sea—Bridging Gaps between National, Individual and Common Interests, edited by Gemma Andreone

oa_id: https://openalex.org/W2889159612
oa_title: Vulnerability of tropical Pacific fisheries and aquaculture to climate change : summary for Pacific island countries and territories / by Johann D. Bell ... [et al.].



KeyboardInterrupt: 

In [55]:
df_results2 = pd.json_normalize(df_results)

# merge with df
merged_df = pd.merge(df,df_results2, left_index=True,right_index=True)


In [58]:
merged_df_ones = merged_df[merged_df['count_y']== 1.0]
print(Fore.LIGHTCYAN_EX + f" length of those with 1 match: {len(merged_df_ones)}")

merged_df_morethans = merged_df[merged_df['count_y']> 1.0]
print(Fore.LIGHTGREEN_EX + f"count of those more than 1: {len(merged_df_morethans)}")

 length of those with 1 match: 112
count of those more than 1: 150


In [68]:
# calc levenshtein seqratio distance
import Levenshtein

text1 = "amap assessment 2015 human health in the arctic"
text2 = "AMAP assessment 2015: human health in the Arctic"
Levenshtein.seqratio(text1,text2)

merged_df_ones['levenshtein'] = merged_df_ones.apply(lambda x: Levenshtein.seqratio(x['title'],x['oa_title']),axis=1)

<positron-console-cell-68>:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


In [69]:
# save out. 
merged_df_ones.to_csv('merged_df_ones.txt', sep="\t",encoding="utf-8")

In [72]:
merged_df_morethans['levenshtein'] = merged_df_morethans.apply(lambda x: Levenshtein.seqratio(x['title'],x['oa_title']),axis=1)
merged_df_morethans.to_csv('merged_df_morethans.txt', sep="\t", encoding="utf-8")

<positron-console-cell-72>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


## manual matching to confirm
rather than rely solely on levenshtein scores, I manaully checked the titles in the excel file 'IPCC_Report_References_dois_v2.xlsx' on the 5.1 sharedrive:
- https://dalu-my.sharepoint.com/:x:/r/personal/ph818849_dal_ca/Documents/qsslab/projects/tca_wp5/wp5_1_dataset/IPCC_Report_References_dois_v2.xlsx?d=w5a91c4f1c4cc49d994e0c0e2ee17e241&csf=1&web=1&e=xkNvjG

## removals
- [x] manual review merged_df_ones
    - [x] based on manual comparison, I am removing the following indcies: 530, 542
- [ ] manual review merged_df_morethans
    - [ ] removed....


In [70]:
# remove indices 530, 542
merged_df_ones = merged_df_ones.drop(index=[530,542])